In [3]:
!unzip "archive (3).zip"
!ls -lh

Archive:  archive (3).zip
warning [archive (3).zip]:  5645343 extra bytes at beginning or within zipfile
  (attempting to process anyway)
file #1:  bad zipfile offset (local header sig):  5645343
  (attempting to re-compensate)
  inflating: online_retail.csv       
  error:  invalid compressed data to inflate
error: not enough memory for bomb detection
total 19M
-rw-r--r-- 1 root root  13M Oct  6 10:16 'archive (3).zip'
-rw-r--r-- 1 root root 6.2M Jan 20  2023  online_retail.csv
drwxr-xr-x 1 root root 4.0K Oct  2 13:50  sample_data


In [4]:
import pandas as pd
import datetime as dt

# 1. Load the dataset directly from UCI (This will take 1-2 minutes to download)
print("Downloading dataset directly from UCI...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df = pd.read_excel(url)

# 2. Clean the data (remove missing IDs and cancelled orders)
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.contains('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# 3. Calculate Total Spend per transaction
df['TotalSpend'] = df['Quantity'] * df['UnitPrice']

print("Data loaded and cleaned successfully!")
df.head()

Data loaded and cleaned successfully!


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpend
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import datetime as dt

# 1. Establish reference snapshot date (1 day after the latest invoice in dataset)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Aggregate per CustomerID to compute Recency, Frequency, and Monetary values
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Days since last order
    'InvoiceNo': 'nunique',                                  # Count of distinct orders
    'TotalSpend': 'sum'                                      # Total monetary spend
}).reset_index()

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSpend': 'Monetary'
}, inplace=True)

# 3. Standardize RFM features for K-Means (log transform + standard scale to handle skew)
import numpy as np
rfm_log = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# 4. Fit K-Means with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 5. Profile clusters by their median values to map them to marketing segments
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'median',
    'Frequency': 'median',
    'Monetary': 'median'
}).reset_index()

print("Cluster Profiles (Medians):")
print(cluster_summary)

rfm.head()

Cluster Profiles (Medians):
   Cluster  Recency  Frequency  Monetary
0        0     17.0        2.0   471.700
1        1      8.0       10.0  3733.870
2        2     56.0        4.0  1345.620
3        3    177.0        1.0   298.255


,CustomerID,Recency,Frequency,Monetary,Cluster
0,12346.0,326,1,77183.60,2
1,12347.0,2,7,4310.00,1
2,12348.0,75,4,1797.24,2
3,12349.0,19,1,1757.55,0
4,12350.0,310,1,334.40,3


In [6]:
# 1. Map cluster numbers to marketing segments
segment_map = {
    1: 'Champions',
    2: 'At-Risk',
    0: 'New / Promising',
    3: 'Lost'
}
rfm['Segment'] = rfm['Cluster'].map(segment_map)

# 2. View segment breakdown
print(rfm['Segment'].value_counts())

# 3. Export to CSV for Power BI visualization
rfm.to_csv('customer_rfm_segments.csv', index=False)
print("\nExported 'customer_rfm_segments.csv' successfully!")

Segment
Lost               1612
At-Risk            1173
New / Promising     837
Champions           716
Name: count, dtype: int64

Exported 'customer_rfm_segments.csv' successfully!
